# Treino Melhorado - MIQR-CC Dataset

Classificação automática de imagens CPRE em 4 classes:
- `Biliary_Leaks`, `Lithiasis`, `Normal`, `Stricture`

**Modelos treinados:**
1. ResNet-50 — baseline de comparação
2. EfficientNet-B3 — principal aposta
3. Ensemble (ResNet-50 + EfficientNet-B3) — arquitetura híbrida

**Técnicas aplicadas:**
- Transfer Learning + Fine-tuning em 2 fases
- CrossEntropyLoss ponderada por classe
- WeightedRandomSampler (oversampling)
- CosineAnnealingLR + Early Stopping
- Grad-CAM para interpretabilidade

**Baseline de referência:** F1-macro = 0.738 (EfficientNet-B7, Martins et al.)

In [5]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 
                'grad-cam', 'livelossplot', 'torchmetrics', 'timm', '--quiet'])

CompletedProcess(args=['c:\\Users\\Filipa Pino\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'pip', 'install', 'grad-cam', 'livelossplot', 'torchmetrics', 'timm', '--quiet'], returncode=0)

In [6]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)
from torchmetrics.classification import MulticlassF1Score
from livelossplot import PlotLosses
from livelossplot.outputs import MatplotlibPlot

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')

PyTorch: 2.11.0+cpu
CUDA disponível: False


---
## 1. Configuração

In [7]:
# ============================================================
# CONFIGURAÇÃO — ajusta o DATASET_DIR ao teu ambiente
# ============================================================
DATASET_DIR = './dataset_processed'   # saída do preprocessing_improved.ipynb
MODEL_DIR   = './models'
BATCH_SIZE  = 16
NUM_WORKERS = 4
SEED        = 42

# Fase 1: treinar só a cabeça
EPOCHS_PHASE1 = 10
LR_PHASE1     = 1e-3

# Fase 2: fine-tuning completo
EPOCHS_PHASE2      = 50
LR_PHASE2          = 1e-5
EARLY_STOP_PATIENCE = 10

# Modelos a treinar: (nome timm, input_size)
MODELOS = [
    ('resnet50',        224),
    ('efficientnet_b3', 300),
]
# ============================================================

os.makedirs(MODEL_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


---
## 2. Funções auxiliares — Transforms, DataLoaders, Modelo

In [8]:
def build_transforms(input_size, augment=True):
    """Cria os transforms de treino (com augmentation) ou val/test."""
    if augment:
        return transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((input_size, input_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(degrees=15),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.3, contrast=0.4),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    return transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])


def build_dataloaders(input_size, batch_size, num_workers):
    """Cria os DataLoaders com WeightedRandomSampler para oversampling."""
    train_ds = ImageFolder(os.path.join(DATASET_DIR, 'train'),
                           transform=build_transforms(input_size, augment=True))
    val_ds   = ImageFolder(os.path.join(DATASET_DIR, 'val'),
                           transform=build_transforms(input_size, augment=False))
    test_ds  = ImageFolder(os.path.join(DATASET_DIR, 'test'),
                           transform=build_transforms(input_size, augment=False))

    # Oversampling — WeightedRandomSampler
    train_labels   = [label for _, label in train_ds.samples]
    counts         = np.bincount(train_labels)
    sample_weights = torch.tensor([1.0 / counts[l] for l in train_labels], dtype=torch.float32)
    sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader, test_loader, train_ds.classes, counts


def build_model(model_name, num_classes, freeze_backbone=True):
    """Cria modelo pré-treinado com timm. Congela backbone se pedido."""
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    if freeze_backbone:
        for name, param in model.named_parameters():
            if 'classifier' not in name and 'fc' not in name:
                param.requires_grad = False
    return model


def unfreeze_model(model):
    for param in model.parameters():
        param.requires_grad = True
    return model

---
## 3. Funções de treino e avaliação

In [9]:
def train_one_epoch(model, loader, optimizer, loss_fn, f1_metric):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        all_preds.append(outputs.argmax(dim=1))
        all_labels.append(labels)
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    return running_loss / len(loader.dataset), f1_metric(all_preds, all_labels).item()


@torch.no_grad()
def evaluate(model, loader, loss_fn, f1_metric):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        running_loss += loss_fn(outputs, labels).item() * inputs.size(0)
        all_preds.append(outputs.argmax(dim=1))
        all_labels.append(labels)
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    return running_loss / len(loader.dataset), f1_metric(all_preds, all_labels).item()


def train_phase(model, train_loader, val_loader, optimizer, loss_fn,
                epochs, phase_name, save_path, num_classes, patience=10):
    f1_metric = MulticlassF1Score(num_classes=num_classes, average='macro').to(device)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    liveloss  = PlotLosses(outputs=[MatplotlibPlot(figpath=f"{save_path}_{phase_name}.png")])
    best_f1, no_improve = -1, 0
    history = []

    print(f"\n{'='*50}\n {phase_name}\n{'='*50}")

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, loss_fn, f1_metric)
        val_loss,   val_f1   = evaluate(model, val_loader, loss_fn, f1_metric)
        scheduler.step()
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'train_f1': train_f1, 'val_f1': val_f1})
        liveloss.update({'loss': train_loss, 'val_loss': val_loss,
                         'F1': train_f1, 'val_F1': val_f1})
        liveloss.send()
        print(f"Época {epoch:3d}/{epochs} | Loss {train_loss:.4f}/{val_loss:.4f} | "
              f"F1 {train_f1:.4f}/{val_f1:.4f} | {time.time()-t0:.1f}s")
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), save_path)
            no_improve = 0
            print(f"  ✓ Melhor F1-val: {best_f1:.4f} — guardado")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping na época {epoch}")
                break

    print(f"Melhor F1-val: {best_f1:.4f}")
    return pd.DataFrame(history)


@torch.no_grad()
def avaliar_teste(model, test_loader, class_names, model_name):
    """Avalia no teste e mostra métricas + matriz de confusão."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for inputs, labels in test_loader:
        outputs = model(inputs.to(device))
        probs   = torch.softmax(outputs, dim=1).cpu().numpy()
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    f1  = f1_score(all_labels, all_preds, average='macro')
    acc = accuracy_score(all_labels, all_preds)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    except:
        auc = float('nan')

    print(f"\n{'='*40}")
    print(f" {model_name} — TESTE")
    print(f"{'='*40}")
    print(f" Accuracy : {acc:.4f}")
    print(f" F1-macro : {f1:.4f}  (baseline: 0.738)")
    print(f" AUC-ROC  : {auc:.4f}")
    print(f"{'='*40}")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # Matriz de confusão
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predito')
    plt.ylabel('Real')
    plt.title(f'{model_name} — Matriz de Confusão\nF1-macro: {f1:.4f}')
    plt.tight_layout()
    cm_path = os.path.join(MODEL_DIR, f'{model_name}_cm.png')
    plt.savefig(cm_path, dpi=150)
    plt.show()

    return {'model': model_name, 'f1_macro': f1, 'accuracy': acc, 'auc_roc': auc,
            'preds': all_preds, 'labels': all_labels, 'probs': all_probs}

---
## 4. Treino dos modelos

Corre sequencialmente: **ResNet-50** e **EfficientNet-B3**.
Cada um passa por fine-tuning em 2 fases.

In [10]:
resultados = {}
modelos_treinados = {}

for model_name, input_size in MODELOS:
    print(f"\n{'#'*60}")
    print(f"  MODELO: {model_name.upper()}")
    print(f"{'#'*60}")

    save_path = os.path.join(MODEL_DIR, f'{model_name}_best.pth')

    # DataLoaders com o input_size correto para cada modelo
    train_loader, val_loader, test_loader, class_names, counts = build_dataloaders(
        input_size, BATCH_SIZE, NUM_WORKERS
    )
    NUM_CLASSES = len(class_names)

    # Class weights para CrossEntropyLoss
    total = len(train_loader.dataset)
    class_weights = torch.tensor(
        [total / (NUM_CLASSES * c) for c in counts], dtype=torch.float32
    ).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    # --- Fase 1: backbone congelado ---
    model = build_model(model_name, NUM_CLASSES, freeze_backbone=True).to(device)
    optimizer_p1 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE1
    )
    h1 = train_phase(model, train_loader, val_loader, optimizer_p1, loss_fn,
                     EPOCHS_PHASE1, f'{model_name}_Fase1', save_path, NUM_CLASSES, EARLY_STOP_PATIENCE)

    # --- Fase 2: fine-tuning completo ---
    model.load_state_dict(torch.load(save_path, map_location=device))
    model = unfreeze_model(model)
    optimizer_p2 = torch.optim.Adam(model.parameters(), lr=LR_PHASE2)
    h2 = train_phase(model, train_loader, val_loader, optimizer_p2, loss_fn,
                     EPOCHS_PHASE2, f'{model_name}_Fase2', save_path, NUM_CLASSES, EARLY_STOP_PATIENCE)

    # --- Avaliação no teste ---
    model.load_state_dict(torch.load(save_path, map_location=device))
    res = avaliar_teste(model, test_loader, class_names, model_name)
    resultados[model_name] = res
    modelos_treinados[model_name] = (model, input_size)

    # Curvas de treino
    h2['epoch'] = h2['epoch'] + len(h1)
    history = pd.concat([h1, h2], ignore_index=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history['epoch'], history['train_loss'], label='Treino')
    axes[0].plot(history['epoch'], history['val_loss'],   label='Validação')
    axes[0].axvline(len(h1), linestyle='--', color='gray', label='Início Fase 2')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[1].plot(history['epoch'], history['train_f1'], label='Treino')
    axes[1].plot(history['epoch'], history['val_f1'],   label='Validação')
    axes[1].axhline(0.738, linestyle=':', color='red', label='Baseline (0.738)')
    axes[1].axvline(len(h1), linestyle='--', color='gray', label='Início Fase 2')
    axes[1].set_title('F1-score macro')
    axes[1].legend()
    plt.suptitle(f'{model_name} — Curvas de treino')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, f'{model_name}_history.png'), dpi=150)
    plt.show()


############################################################
  MODELO: RESNET50
############################################################


FileNotFoundError: [WinError 3] O sistema não conseguiu localizar o caminho especificado: './dataset_processed\\train'

---
## 5. Ensemble — Arquitetura Híbrida

Combina as probabilidades do ResNet-50 e EfficientNet-B3 por soft voting (média).

In [ ]:
print("\n" + "#"*60)
print("  ENSEMBLE (ResNet-50 + EfficientNet-B3)")
print("#"*60)

# Carrega os dois modelos treinados
model_res = build_model('resnet50', NUM_CLASSES, freeze_backbone=False).to(device)
model_res.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'resnet50_best.pth'), map_location=device))
model_res.eval()

model_efn = build_model('efficientnet_b3', NUM_CLASSES, freeze_backbone=False).to(device)
model_efn.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'efficientnet_b3_best.pth'), map_location=device))
model_efn.eval()

# Precisamos de dois test_loaders com input_sizes diferentes
_, _, test_loader_224, class_names, _ = build_dataloaders(224, BATCH_SIZE, NUM_WORKERS)
_, _, test_loader_300, class_names, _ = build_dataloaders(300, BATCH_SIZE, NUM_WORKERS)

# Recolhe probabilidades de cada modelo
def get_probs(model, loader):
    probs_all = []
    labels_all = []
    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs.to(device))
            probs_all.append(torch.softmax(outputs, dim=1).cpu().numpy())
            labels_all.append(labels.numpy())
    return np.vstack(probs_all), np.concatenate(labels_all)

probs_res, labels = get_probs(model_res, test_loader_224)
probs_efn, _      = get_probs(model_efn, test_loader_300)

# Soft voting — média das probabilidades
probs_ensemble = (probs_res + probs_efn) / 2
preds_ensemble = probs_ensemble.argmax(axis=1)

f1_ens  = f1_score(labels, preds_ensemble, average='macro')
acc_ens = accuracy_score(labels, preds_ensemble)
try:
    auc_ens = roc_auc_score(labels, probs_ensemble, multi_class='ovr', average='macro')
except:
    auc_ens = float('nan')

print(f"\nEnsemble — F1-macro : {f1_ens:.4f}  (baseline: 0.738)")
print(f"Ensemble — Accuracy : {acc_ens:.4f}")
print(f"Ensemble — AUC-ROC  : {auc_ens:.4f}")
print(classification_report(labels, preds_ensemble, target_names=class_names))

# Matriz de confusão
cm = confusion_matrix(labels, preds_ensemble)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title(f'Ensemble — Matriz de Confusão\nF1-macro: {f1_ens:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'ensemble_cm.png'), dpi=150)
plt.show()

resultados['ensemble'] = {'model': 'ensemble', 'f1_macro': f1_ens,
                          'accuracy': acc_ens, 'auc_roc': auc_ens}

---
## 6. Tabela comparativa final

In [ ]:
# Adiciona a baseline do artigo
resultados['baseline_efn_b7'] = {'model': 'EfficientNet-B7 (baseline)',
                                  'f1_macro': 0.738, 'accuracy': None, 'auc_roc': None}

df_res = pd.DataFrame([
    {
        'Modelo': v['model'],
        'F1-macro': f"{v['f1_macro']:.4f}",
        'Accuracy': f"{v['accuracy']:.4f}" if v['accuracy'] else '—',
        'AUC-ROC':  f"{v['auc_roc']:.4f}"  if v['auc_roc']  else '—',
    }
    for v in resultados.values()
])

print("\n" + "="*60)
print(" COMPARAÇÃO FINAL")
print("="*60)
print(df_res.to_string(index=False))
print("="*60)

---
## 7. Grad-CAM — Interpretabilidade (obrigatório)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def generate_gradcam(model, image_path, input_size, target_layer, true_label=None):
    cam = GradCAM(model=model, target_layers=[target_layer])
    img_pil = Image.open(image_path).convert('RGB').resize((input_size, input_size))
    img_np  = np.array(img_pil).astype(np.float32) / 255.0
    tensor  = build_transforms(input_size, augment=False)(Image.open(image_path)).unsqueeze(0).to(device)

    with torch.no_grad():
        output   = model(tensor)
        pred_idx = output.argmax(dim=1).item()
        pred_prob = torch.softmax(output, dim=1)[0, pred_idx].item()

    grayscale = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred_idx)])[0]
    cam_img   = show_cam_on_image(img_np, grayscale, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(cam_img)
    title = f'Pred: {class_names[pred_idx]} ({pred_prob:.2f})'
    if true_label is not None:
        title += f'\nReal: {class_names[true_label]}'
        axes[1].set_title(title, color='green' if pred_idx == true_label else 'red')
    else:
        axes[1].set_title(title)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


# Usa o EfficientNet-B3 para Grad-CAM (melhor candidato)
model_efn.eval()
target_layer = model_efn.blocks[-1]

print("Grad-CAM — 2 exemplos por classe (EfficientNet-B3)\n")
for class_idx, class_name in enumerate(class_names):
    class_dir = os.path.join(DATASET_DIR, 'test', class_name)
    images    = [f for f in os.listdir(class_dir) if f.endswith('.png')]
    samples   = random.sample(images, min(2, len(images)))
    print(f"--- {class_name} ---")
    for img_name in samples:
        generate_gradcam(model_efn, os.path.join(class_dir, img_name),
                         300, target_layer, true_label=class_idx)